# Unit Tests
Run all backend tests from this notebook after each important backend change.

In [ ]:
import json
import unittest

import backend.server as server
from backend.models import create_connection_state, create_match, create_user_session
from backend.protocol import ProtocolError, decode_message, encode_message


In [ ]:
class DummySocket:
    def __init__(self):
        self.sent = []

    def sendall(self, data):
        self.sent.append(data)

    def shutdown(self, how):
        return None

    def close(self):
        return None


def parse_sent_messages(sock):
    parsed = []
    for raw in sock.sent:
        line = raw.decode('utf-8').strip()
        parsed.append(json.loads(line))
    return parsed


def message_types(sock):
    return [message['type'] for message in parse_sent_messages(sock)]


class TestProtocol(unittest.TestCase):
    def test_encode_message_has_newline_and_shape(self):
        raw = encode_message('LOGIN', {'username': 'Ali'})
        self.assertTrue(raw.endswith(b'\n'))
        parsed = json.loads(raw.decode('utf-8'))
        self.assertEqual(parsed['type'], 'LOGIN')
        self.assertEqual(parsed['payload'], {'username': 'Ali'})

    def test_decode_message_valid(self):
        parsed = decode_message(b'{"type":"WAITING","payload":{}}')
        self.assertEqual(parsed, {'type': 'WAITING', 'payload': {}})

    def test_decode_message_rejects_invalid_messages(self):
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN"')
        with self.assertRaises(ProtocolError):
            decode_message(b'{"payload":{}}')
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN","payload":[]}')


class TestServer(unittest.TestCase):
    def setUp(self):
        server.STATE = create_connection_state()
        server.RUNNING.clear()

    def register_user(self, username, port):
        sock = DummySocket()
        session = create_user_session(sock, ('127.0.0.1', port))
        server.handle_login(session, {'username': username})
        return session, sock

    def test_login_uniqueness(self):
        _, first_sock = self.register_user('Ali', 5001)
        second_sock = DummySocket()
        second_session = create_user_session(second_sock, ('127.0.0.1', 5002))
        server.handle_login(second_session, {'username': 'Ali'})

        self.assertIn('LOGIN_OK', message_types(first_sock))
        self.assertIsNone(second_session['username'])
        self.assertEqual(parse_sent_messages(second_sock)[0]['type'], 'LOGIN_REJECT')

    def test_waiting_and_watch_transitions(self):
        session, sock = self.register_user('Ali', 5001)

        server.dispatch_message(session, {'type': 'WAITING', 'payload': {}})
        self.assertIn('Ali', server.STATE['waiting_players'])

        server.dispatch_message(session, {'type': 'WATCH_MATCH', 'payload': {}})
        self.assertNotIn('Ali', server.STATE['waiting_players'])
        self.assertIn('Ali', server.STATE['spectators'])

        types = message_types(sock)
        self.assertIn('WAITING', types)
        self.assertIn('WATCH_MATCH', types)

    def test_challenge_and_accept_create_match(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        maya_session, maya_sock = self.register_user('Maya', 5002)

        server.handle_challenge_player(ali_session, {'target': 'Maya'})
        self.assertEqual(server.STATE['pending_challenges'].get('Maya'), 'Ali')
        self.assertIn('CHALLENGE_PLAYER', message_types(ali_sock))
        self.assertIn('CHALLENGE_RECEIVED', message_types(maya_sock))

        server.handle_challenge_accept(maya_session, {'from': 'Ali'})
        self.assertIsNotNone(server.STATE['active_match'])
        players = set(server.STATE['active_match']['players'])
        self.assertEqual(players, {'Ali', 'Maya'})
        self.assertIn('MATCH_START', message_types(ali_sock))
        self.assertIn('MATCH_START', message_types(maya_sock))

    def test_spectator_promoted_to_player_gets_single_player_start(self):
        ali_session, ali_sock = self.register_user('Ali', 5001)
        maya_session, maya_sock = self.register_user('Maya', 5002)

        server.dispatch_message(ali_session, {'type': 'WATCH_MATCH', 'payload': {}})
        server.dispatch_message(maya_session, {'type': 'WATCH_MATCH', 'payload': {}})
        self.assertIn('Ali', server.STATE['spectators'])
        self.assertIn('Maya', server.STATE['spectators'])

        server.handle_challenge_player(ali_session, {'target': 'Maya'})
        server.handle_challenge_accept(maya_session, {'from': 'Ali'})

        self.assertNotIn('Ali', server.STATE['spectators'])
        self.assertNotIn('Maya', server.STATE['spectators'])

        ali_starts = [m for m in parse_sent_messages(ali_sock) if m['type'] == 'MATCH_START']
        maya_starts = [m for m in parse_sent_messages(maya_sock) if m['type'] == 'MATCH_START']
        self.assertEqual(len(ali_starts), 1)
        self.assertEqual(len(maya_starts), 1)
        self.assertFalse(ali_starts[0]['payload']['spectator'])
        self.assertFalse(maya_starts[0]['payload']['spectator'])

    def test_input_updates_pending_direction(self):
        ali_session, _ = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)
        match, _ = server.create_and_start_match('Ali', 'Maya')
        self.assertIsNotNone(match)

        server.handle_input(ali_session, {'direction': 'UP'})
        self.assertEqual(server.STATE['active_match']['snakes']['Ali']['pending_direction'], 'UP')

        bad_sock = DummySocket()
        bad_session = create_user_session(bad_sock, ('127.0.0.1', 5010))
        bad_session['username'] = 'Ghost'
        server.handle_input(bad_session, {'direction': 'LEFT'})
        self.assertEqual(parse_sent_messages(bad_sock)[0]['type'], 'ERROR')

    def test_advance_tick_can_finish_on_timer(self):
        config = server.get_match_config()
        match = create_match(99, 'Ali', 'Maya', config)
        match['remaining_ticks'] = 1
        match['obstacles'] = []
        match['pies'] = []
        match['snakes']['Ali']['body'] = [(5, 5), (4, 5), (3, 5)]
        match['snakes']['Maya']['body'] = [(20, 5), (21, 5), (22, 5)]
        match['snakes']['Ali']['direction'] = 'RIGHT'
        match['snakes']['Ali']['pending_direction'] = 'RIGHT'
        match['snakes']['Maya']['direction'] = 'LEFT'
        match['snakes']['Maya']['pending_direction'] = 'LEFT'
        match['snakes']['Ali']['health'] = 80
        match['snakes']['Maya']['health'] = 70

        server.advance_match_one_tick(match, config)

        self.assertEqual(match['status'], 'ended')
        self.assertEqual(match['reason'], 'timer_end')
        self.assertEqual(match['winner'], 'Ali')

    def test_disconnect_marks_match_ended(self):
        ali_session, _ = self.register_user('Ali', 5001)
        self.register_user('Maya', 5002)
        match, _ = server.create_and_start_match('Ali', 'Maya')
        self.assertEqual(match['status'], 'running')

        server.disconnect_session(ali_session)

        active = server.STATE['active_match']
        self.assertIsNotNone(active)
        self.assertEqual(active['status'], 'ended')
        self.assertEqual(active['reason'], 'player_disconnected')
        self.assertEqual(active['winner'], 'Maya')


In [ ]:
suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestProtocol))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestServer))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

if not result.wasSuccessful():
    raise AssertionError('Some unit tests failed')
